In [33]:

import os
import csv
import torch
import pandas as pd

from torch import nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset ,DataLoader
from torchvision.io import read_image
from torchvision import transforms

from pathlib import Path

In [30]:
#DEFINING CONSTATNTS

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATASET_DIR = PROJECT_DIR / "data" / "raw"
ANNOTATIONS = PROJECT_DIR / "data" / "labels.csv"

IMG_SIZE = (300, 300)
BATCH_SIZE = 32
EPOCHS = 10

In [17]:
def create_labels_csv(csv_file_path="labels.csv") -> pd.DataFrame:
    """
    Creates a labels.csv file for the NSL image classification dataset.

    CSV columns:
        image_path, label, background
    """
    try:
        if not DATASET_DIR.exists():
            raise FileNotFoundError(f"Dataset not found: {DATASET_DIR}")

        data = []

        for background_dir in sorted(DATASET_DIR.iterdir()):

            if not background_dir.is_dir():
                continue

            for class_dir in sorted(background_dir.iterdir()):

                if not class_dir.is_dir():
                    continue

                class_name = class_dir.name

                for image_path in sorted(class_dir.glob("*.jpg")):

                    relative_path = image_path.relative_to(PROJECT_DIR)

                    data.append([
                        relative_path.as_posix(),
                        class_name,
                        background_dir.name
                    ])

        df = pd.DataFrame(
            data,
            columns=["image_path", "label", "background"]
        )

        # Save CSV
        csv_path = Path(csv_file_path)

        if not csv_path.is_absolute():
            csv_path = PROJECT_DIR / csv_path

        csv_path.parent.mkdir(parents=True, exist_ok=True)

        df.to_csv(csv_path, index=False)

        print(f"CSV created successfully: {csv_path}")
        print(f"Total images: {len(df)}")
        print(f"Total classes: {df['label'].nunique()}")
        print(f"Background categories: {df['background'].nunique()}")

        return df

    except Exception as e:

        print(f"Couldn't create CSV file. Error: {e}")

        return pd.DataFrame()


labels_df = create_labels_csv("data/processed/labels.csv")

display(labels_df.head())


CSV created successfully: c:\Users\apili\Desktop\Academics\Sixth Sem\MLIS\MLIS-INDV-ASSING\data\processed\labels.csv
Total images: 54000
Total classes: 36
Background categories: 2


,image_path,label,background
0,data/raw/NSL/plain_background/0/0.jpg,0,plain_background
1,data/raw/NSL/plain_background/0/1.jpg,0,plain_background
2,data/raw/NSL/plain_background/0/10.jpg,0,plain_background
3,data/raw/NSL/plain_background/0/100.jpg,0,plain_background
4,data/raw/NSL/plain_background/0/101.jpg,0,plain_background


In [25]:
class CustomImageDataset(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = self.img_dir / self.img_labels.iloc[idx]["image_path"]

        image = read_image(str(img_path))
        label = int(self.img_labels.iloc[idx]["label"])

        if self.transform:
            image = self.transform(image)

        if self.target_transform:
            label = self.target_transform(label)

        return image, label

In [62]:
images_path = Path("notebooks/dataset")
annotations = Path("data/labels.csv")

transform = transforms.Compose([
   transforms.Resize(IMG_SIZE),
    transforms.Grayscale(num_output_channels=1),
    transforms.ConvertImageDtype(torch.float32),
    transforms.Normalize((0.5,), (0.5,))
])

In [63]:
train_dataset = CustomImageDataset(
    annotations_file=ANNOTATIONS,
    img_dir=PROJECT_DIR,
    transform=transform
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

In [73]:

class CNN(nn.Module):
    def __init__(self, num_classes=36):
        super().__init__()

        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)

        # Max pooling
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Adaptive pooling
        self.adaptive_pool = nn.AdaptiveAvgPool2d((7, 7))

        # Fully connected layers
        self.fc1 = nn.Linear(64 * 7 * 7, 512)
        self.fc2 = nn.Linear(512, num_classes)

        # Dropout
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = self.adaptive_pool(x)
        x = torch.flatten(x, start_dim=1)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)

        # Raw logits for CrossEntropyLoss
        x = self.fc2(x)

        return x


In [74]:
NUM_CLASSES = 36

model = CNN(num_classes=NUM_CLASSES)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in train_dataloader:

        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels.long())

        # Backward pass
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(
        f'Epoch [{epoch+1}/{EPOCHS}], '
        f'Loss: {running_loss/len(train_dataloader):.4f}'
    )